# PPLM Hyperparameter Grid Search

This notebook performs the hyperparameter search used to select the inference-time configuration for the PPLM baseline.

A fixed, balanced calibration subset is constructed from the In-Domain Evaluation Prompt Matrix using 20 topics, each paired with all six CEFR target levels, resulting in **120 generation conditions**.

The search evaluates combinations of:

- PPLM perturbation step size; and
- number of gradient update steps applied at each generation step.

For every configuration, generations are produced with the frozen `meta-llama/Llama-3.1-8B-Instruct` backbone and the CEFR latent classifier:

`MohammadKhosravi/llama3.1-8b-cefr-steering-1layer-head-ordinal-universal`

Generated texts are evaluated using the primary CEFR evaluator:

`MohammadKhosravi/roberta-large-cefr-classifier-JointLoss`

The search initially evaluates 50 configurations and is subsequently extended to 70 configurations. Results are ranked primarily by Strict Accuracy, with Adjacent Accuracy and additional linguistic diagnostics retained for analysis.

## Grid search PPLM Hyperparams (50 States)

In [ ]:
!pip install -q transformers torch datasets tqdm huggingface_hub accelerate scikit-learn textstat spacy
!python -m spacy download en_core_web_sm

import os
import gc
import torch
import spacy
import textstat
import pandas as pd
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
from huggingface_hub import login, hf_hub_download
from google.colab import drive


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 110.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 132.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# ==========================================
# 0. CONFIGURATION & GOOGLE DRIVE SETUP
# ==========================================
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
INPUT_CSV_PATH = "/content/drive/MyDrive/Your_Path/in_domain_evaluation_prompt_matrix.csv"
BASE_OUTPUT_DIR = "/content/drive/MyDrive/Your_Path/pplm_grid_search_results"
GRID_SEARCH_DIR = os.path.join(BASE_OUTPUT_DIR, "GRID_SEARCH")
os.makedirs(GRID_SEARCH_DIR, exist_ok=True)

CALIBRATION_CSV_PATH = os.path.join(GRID_SEARCH_DIR, "grid_search_calibration_subset.csv")

hf_token = "HF_Token"
login(token=hf_token)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load NLP structural tools
nlp = spacy.load("en_core_web_sm")

Using device: cuda


In [ ]:
# ==========================================
# 1. GENERATE / LOAD PERMANENT CALIBRATION SPLIT
# ==========================================
df_master = pd.read_csv(INPUT_CSV_PATH)
df_master['cefr'] = df_master['cefr'].astype(str).str.strip().str.upper()

if os.path.exists(CALIBRATION_CSV_PATH):
    print(f"Loading existing permanent 120-prompt calibration subset from: {CALIBRATION_CSV_PATH}")
    df_calibration = pd.read_csv(CALIBRATION_CSV_PATH)
else:
    print("Extracting deterministic 120-prompt calibration subset (20 topics × 6 levels)...")
    # Group by title to identify topics that have variants for all 6 target categories
    grouped = df_master.groupby('topic_title').filter(lambda x: set(["A1", "A2", "B1", "B2", "C1", "C2"]).issubset(set(x['cefr'])))
    unique_topics = grouped['topic_title'].unique()

    if len(unique_topics) < 20:
        raise ValueError(f"Error: Found only {len(unique_topics)} unique topics containing all 6 levels. Expected at least 20.")

    # Pick exactly 20 unique topics deterministically using a fixed random state
    np.random.seed(42)
    selected_topics = np.random.choice(unique_topics, size=20, replace=False)

    # Filter master data to pull matching rows
    df_filtered = df_master[df_master['topic_title'].isin(selected_topics)]

    # Isolate exactly 1 row per CEFR level for each of the selected 20 topics to ensure exactly 120 rows
    final_rows = []
    for topic in selected_topics:
        topic_df = df_filtered[df_filtered['topic_title'] == topic]
        for level in ["A1", "A2", "B1", "B2", "C1", "C2"]:
            matched_row = topic_df[topic_df['cefr'] == level].head(1)
            final_rows.append(matched_row)

    df_calibration = pd.concat(final_rows, ignore_index=True)
    df_calibration.to_csv(CALIBRATION_CSV_PATH, index=False)
    print(f"Saved locked calibration split validation file to Drive: {CALIBRATION_CSV_PATH}")

print(f"Calibration Dataset Check -> Length: {len(df_calibration)} rows (Expected: 120)")


Extracting deterministic 120-prompt calibration subset (20 topics × 6 levels)...
Saved locked calibration split validation file to Drive: /content/drive/MyDrive/Mohammd_Thesis/Results/PPLM_Pure/GRID_SEARCH/grid_search_calibration_subset.csv
Calibration Dataset Check -> Length: 120 rows (Expected: 120)


In [ ]:
# ==========================================
# 2. LOAD LLM & TOKENIZER
# ==========================================
print("Loading Llama-3.1-8B-Instruct...")
model_id = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>\n") if tokenizer.convert_tokens_to_ids("<|eot_id|>\n") is not None else tokenizer.convert_tokens_to_ids("<|eot_id|>. ")
if eot_id is None:
    eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
eos_ids = [tokenizer.eos_token_id]
if eot_id is not None:
    eos_ids.append(eot_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, device_map="auto"
)
model.eval()

Loading Llama-3.1-8B-Instruct...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
  

In [ ]:
# ==========================================
# 3. LOAD PPLM STEERING HEAD
# ==========================================
# =========================================================
# LOAD THE PPLM CEFR LATENT CLASSIFIER
# =========================================================

print("Loading PPLM CEFR latent classifier...")

class CEFRLatentClassifier(nn.Module):
    def __init__(self, input_dim=4096, num_classes=6):
        super().__init__()
        self.network = nn.Sequential(
            nn.Dropout(p=0.35),
            nn.Linear(input_dim, num_classes)
        )

    def forward(self, x):
        return self.network(x)

target_repo_id = (
    "MohammadKhosravi/"
    "llama3.1-8b-cefr-steering-1layer-head-ordinal-universal"
)

steering_head = CEFRLatentClassifier().to(device)

weights_path = hf_hub_download(
    repo_id=target_repo_id,
    filename="cefr_steering_head_universal_ordinal.pt"
)

steering_head.load_state_dict(
    torch.load(weights_path, map_location=device)
)

steering_head.eval()

criterion = nn.CrossEntropyLoss(reduction="none")

In [ ]:
# ==========================================
# 4. LOAD CUSTOM JOINT LOSS EVALUATOR
# ==========================================
print("Loading Custom JointLoss RoBERTa Evaluator...")
JUDGE_MODEL_ID = "MohammadKhosravi/roberta-large-cefr-classifier-JointLoss"
judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID)
judge_model = AutoModelForSequenceClassification.from_pretrained(JUDGE_MODEL_ID).to(device)
judge_model.eval()

label_map = {"A1": 0, "A2": 1, "B1": 2, "B2": 3, "C1": 4, "C2": 5}
inv_label_map = {0: "A1", 1: "A2", 2: "B1", 3: "B2", 4: "C1", 5: "C2"}

Loading Custom JointLoss RoBERTa Evaluator...


config.json:   0%|          | 0.00/900 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/388 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [ ]:
# ==========================================
# 5. METRIC HELPER FUNCTIONS
# ==========================================
def calculate_mdd(text):
    doc = nlp(text)
    total_dist, tokens = 0, 0
    for token in doc:
        if token.dep_ != "punct":
            total_dist += abs(token.i - token.head.i)
            tokens += 1
    return total_dist / tokens if tokens > 0 else 0.0

def calculate_sentence_drift(text, judge_model, judge_tokenizer):
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents if len(sent.text.strip()) > 10]
    if len(sentences) <= 1: return 0

    inputs = judge_tokenizer(sentences, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        preds = torch.argmax(judge_model(**inputs).logits, dim=-1).cpu().numpy()

    return int(np.max(preds) - np.min(preds))

In [ ]:
# ==========================================
# 6. GRID SEARCH EXECUTION PARAMETERS
# ==========================================
step_sizes = [0.01, 0.02, 0.05, 0.07, 0.09, 0.1, 0.3, 0.5, 0.7, 0.9]
grad_steps_list = [5, 10, 15, 20, 25]

BATCH_SIZE = 4096
temperature = 0.6
repetition_penalty = 1.15
max_new_tokens = 200
top_k = 50

print(f"\n🚀 STARTING GRID SEARCH ROUTINE ({len(step_sizes) * len(grad_steps_list)} total permutations)...")

for step_size in step_sizes:
    for grad_steps in grad_steps_list:

        # Define target folder for this hyperparameter permutation state
        state_folder_name = f"step_size_{step_size}_grad_step_{grad_steps}"
        state_dir = os.path.join(GRID_SEARCH_DIR, state_folder_name)
        os.makedirs(state_dir, exist_ok=True)

        state_csv_path = os.path.join(state_dir, "benchmark_results.csv")
        state_txt_path = os.path.join(state_dir, "metrics_report.txt")
        state_img_path = os.path.join(state_dir, "confusion_matrix.png")

        # Check for resumption strategy compatibility
        if os.path.exists(state_txt_path) and os.path.exists(state_csv_path):
            print(f"⏭️ Skipping configuration state [Step: {step_size} | Grads: {grad_steps}] - Complete.")
            continue

        print(f"\n⚙️ Executing Configuration Matrix State: Step Size = {step_size}, Grad Steps = {grad_steps}")
        results = []

        # Batch generation across our locked 120 calibration rows
        for b_start in range(0, len(df_calibration), BATCH_SIZE):
            batch_df = df_calibration.iloc[b_start : b_start + BATCH_SIZE]
            current_batch_size = len(batch_df)

            batch_prompts, batch_targets, batch_target_ids, batch_topic_ids, batch_titles = [], [], [], [], []

            for _, row in batch_df.iterrows():
                t_cefr = str(row['cefr']).strip().upper()
                batch_targets.append(t_cefr)
                batch_target_ids.append(label_map[t_cefr])
                batch_topic_ids.append(row['topic_id'])
                batch_titles.append(row['topic_title'])

                sys_instr = (
                    f"You are an expert English language teacher demonstrating CEFR proficiency levels. "
                    f"Your task is to write a flawless, grammatically correct text responding to this prompt: '{row['topic_title']}'. "
                    f"The output must serve as a perfect textbook example of strictly {t_cefr} level English. "
                    f"If the requested target level is A1/A2, use very simple vocabulary, short sentences, and primitive structures. "
                    f"If the requested target level is C1/C2, utilize highly advanced vocabulary, idioms, and complex sentence patterns. "
                    f"Write only the direct response. Do not write any meta-commentary, greetings, or conversational pleasantries."
                )
                messages = [{"role": "system", "content": sys_instr}, {"role": "user", "content": row['topic_title']}]
                batch_prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

            inputs = tokenizer(batch_prompts, padding=True, return_tensors="pt").to(device)
            input_ids, attention_mask = inputs.input_ids, inputs.attention_mask

            finished = torch.zeros(current_batch_size, dtype=torch.bool, device=device)
            generated_tokens = [[] for _ in range(current_batch_size)]
            batch_target_ids_tensor = torch.tensor(batch_target_ids, device=device)

            past_key_values = None
            current_attention_mask = attention_mask

            # Autoregressive PPLM Fixed Steering Loop Execution
            for step_idx in range(max_new_tokens):
                model_input = input_ids if step_idx == 0 else next_token_ids.unsqueeze(-1)
                if step_idx > 0:
                    current_attention_mask = torch.cat([current_attention_mask, torch.ones(current_batch_size, 1, dtype=torch.long, device=device)], dim=-1)

                with torch.no_grad():
                    outputs = model(
                        input_ids=model_input, attention_mask=current_attention_mask,
                        past_key_values=past_key_values, use_cache=True, output_hidden_states=True
                    )
                past_key_values = outputs.past_key_values

                hidden_state_prenorm = outputs.hidden_states[-2][:, -1, :].detach().clone().to(torch.float32)
                hidden_state_prenorm.requires_grad_(True)

                for g_step in range(grad_steps):
                    class_logits = steering_head(hidden_state_prenorm)
                    loss_vector = criterion(class_logits, batch_target_ids_tensor)

                    active_mask = (~finished).float()
                    loss = (loss_vector * active_mask).sum()
                    if loss.item() == 0: break

                    loss.backward()

                    with torch.no_grad():
                        raw_grad = hidden_state_prenorm.grad.data
                        grad_norms = torch.norm(raw_grad, p=2, dim=-1, keepdim=True) + 1e-9
                        normalized_grad = raw_grad / grad_norms

                        hidden_state_prenorm.data = hidden_state_prenorm.data - (step_size * normalized_grad * active_mask.unsqueeze(-1))
                        hidden_state_prenorm.grad.zero_()

                # Softmax mapping to vocab logit spaces
                with torch.no_grad():
                    steered_normed = model.model.norm(hidden_state_prenorm.to(torch.bfloat16))
                    lm_logits = model.lm_head(steered_normed)

                    for b_idx in range(current_batch_size):
                        for tok_id in set(generated_tokens[b_idx] + input_ids[b_idx].tolist()):
                            if lm_logits[b_idx, tok_id] > 0: lm_logits[b_idx, tok_id] /= repetition_penalty
                            else: lm_logits[b_idx, tok_id] *= repetition_penalty

                    lm_logits = lm_logits / temperature
                    values, indices = torch.topk(lm_logits, top_k, dim=-1)
                    filtered_logits = torch.full_like(lm_logits, float('-inf')).scatter_(1, indices, values)
                    probs_lm = F.softmax(filtered_logits, dim=-1)

                    next_token_ids = torch.multinomial(probs_lm, num_samples=1).squeeze(-1)

                    for b_idx in range(current_batch_size):
                        if not finished[b_idx]:
                            generated_tokens[b_idx].append(next_token_ids[b_idx].item())
                            if next_token_ids[b_idx].item() in eos_ids:
                                finished[b_idx] = True

                if finished.all(): break

            batch_gen_texts = [tokenizer.decode(generated_tokens[b_idx], skip_special_tokens=True).strip() for b_idx in range(current_batch_size)]

            # Evaluation and structural post-processing across batch partitions
            eval_inputs = judge_tokenizer(batch_gen_texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
            with torch.no_grad():
                eval_preds = torch.argmax(judge_model(**eval_inputs).logits, dim=-1).cpu().numpy()

            for b_idx in range(current_batch_size):
                target_cefr = batch_targets[b_idx]
                predicted_cefr = inv_label_map.get(eval_preds[b_idx], "A1")
                gen_text = batch_gen_texts[b_idx]

                is_strict = int(predicted_cefr == target_cefr)
                is_adjacent = int(abs(label_map[predicted_cefr] - label_map[target_cefr]) <= 1)

                try: flesch_score = textstat.flesch_reading_ease(gen_text)
                except: flesch_score = 0.0

                mdd_score = calculate_mdd(gen_text)
                drift_score = calculate_sentence_drift(gen_text, judge_model, judge_tokenizer)

                results.append({
                    "topic_id": batch_topic_ids[b_idx], "topic_title": batch_titles[b_idx],
                    "target_cefr": target_cefr, "predicted_cefr": predicted_cefr,
                    "strict_match": is_strict, "adjacent_match": is_adjacent,
                    "mdd": round(mdd_score, 2), "readability_flesch": round(flesch_score, 2),
                    "sentence_drift_max": drift_score, "generated_text": gen_text
                })

            del inputs, eval_inputs
            torch.cuda.empty_cache()
            gc.collect()

        # Save current state data tracking matrix frames
        df_state_results = pd.DataFrame(results)
        df_state_results.to_csv(state_csv_path, index=False)

        # Macro analysis calculations
        y_true_labels = df_state_results['target_cefr'].astype(str).str.strip().str.upper()
        y_pred_labels = df_state_results['predicted_cefr'].astype(str).str.strip().str.upper()
        y_true_num = y_true_labels.map(label_map)
        y_pred_num = y_pred_labels.map(label_map)

        strict_acc = df_state_results['strict_match'].mean() * 100
        adj_acc = df_state_results['adjacent_match'].mean() * 100
        mae = mean_absolute_error(y_true_num, y_pred_num)
        avg_mdd = df_state_results['mdd'].mean()
        avg_flesch = df_state_results['readability_flesch'].mean()
        avg_drift = df_state_results['sentence_drift_max'].mean()

        report_text = "="*60 + "\n"
        report_text += f" 📊 METRICS REPORT [Step: {step_size} | Grads: {grad_steps}]\n"
        report_text += "="*60 + "\n"
        report_text += f"Total Processed       : {len(df_state_results)}\n"
        report_text += f"Strict Accuracy       : {strict_acc:.2f}%\n"
        report_text += f"Adjacent Accuracy     : {adj_acc:.2f}%\n"
        report_text += f"Mean Abs Error (MAE)  : {mae:.4f}\n"
        report_text += f"Avg MDD Score         : {avg_mdd:.2f}\n"
        report_text += f"Avg Reading Ease      : {avg_flesch:.2f}\n"
        report_text += f"Avg Sentence Drift    : {avg_drift:.2f} levels\n"
        report_text += "="*60 + "\n\n"

        report_text += "📈 PERFORMANCE BREAKDOWN BY CEFR TARGET LEVEL:\n"
        level_agg = df_state_results.groupby("target_cefr").agg(
            strict_accuracy=("strict_match", lambda x: np.mean(x) * 100),
            avg_mdd=("mdd", "mean"),
            avg_flesch=("readability_flesch", "mean"),
            avg_drift=("sentence_drift_max", "mean")
        ).round(2)
        report_text += level_agg.to_string() + "\n\n"

        report_text += "="*60 + "\n"
        report_text += "📝 DETAILED CLASSIFICATION REPORT:\n"
        cefr_labels = ["A1", "A2", "B1", "B2", "C1", "C2"]
        report_text += classification_report(y_true_labels, y_pred_labels, labels=cefr_labels, zero_division=0)
        report_text += "\n" + "="*60 + "\n"

        with open(state_txt_path, "w") as f:
            f.write(report_text)

        # Plot confusion validation matrices
        cm = confusion_matrix(y_true_labels, y_pred_labels, labels=cefr_labels)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=cefr_labels, yticklabels=cefr_labels, cbar=True, square=True)
        plt.title(f"CEFR Confusion Matrix (Step {step_size} | Grads {grad_steps})", fontsize=11, pad=12)
        plt.xlabel("Predicted CEFR Level", fontsize=9)
        plt.ylabel("Target CEFR Level", fontsize=9)
        plt.tight_layout()
        plt.savefig(state_img_path, dpi=300)
        plt.close()

print("\n🏁 ALL 50 STATES PROCESSED. COMPILING MASTER SUMMARY PROFILE...")



🚀 STARTING GRID SEARCH ROUTINE (50 total permutations)...

⚙️ Executing Configuration Matrix State: Step Size = 0.01, Grad Steps = 5


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



⚙️ Executing Configuration Matrix State: Step Size = 0.01, Grad Steps = 10

⚙️ Executing Configuration Matrix State: Step Size = 0.01, Grad Steps = 15

⚙️ Executing Configuration Matrix State: Step Size = 0.01, Grad Steps = 20

⚙️ Executing Configuration Matrix State: Step Size = 0.01, Grad Steps = 25

⚙️ Executing Configuration Matrix State: Step Size = 0.02, Grad Steps = 5

⚙️ Executing Configuration Matrix State: Step Size = 0.02, Grad Steps = 10

⚙️ Executing Configuration Matrix State: Step Size = 0.02, Grad Steps = 15

⚙️ Executing Configuration Matrix State: Step Size = 0.02, Grad Steps = 20

⚙️ Executing Configuration Matrix State: Step Size = 0.02, Grad Steps = 25

⚙️ Executing Configuration Matrix State: Step Size = 0.05, Grad Steps = 5

⚙️ Executing Configuration Matrix State: Step Size = 0.05, Grad Steps = 10

⚙️ Executing Configuration Matrix State: Step Size = 0.05, Grad Steps = 15

⚙️ Executing Configuration Matrix State: Step Size = 0.05, Grad Steps = 20

⚙️ Executing 

In [ ]:
# ==========================================
# 7. AUTOMATED CENTRAL SUMMARY FILE GENERATION
# ==========================================
summary_records = []
for step_size in step_sizes:
    for grad_steps in grad_steps_list:
        state_folder_name = f"step_size_{step_size}_grad_step_{grad_steps}"
        state_csv_path = os.path.join(GRID_SEARCH_DIR, state_folder_name, "benchmark_results.csv")

        if os.path.exists(state_csv_path):
            df_s = pd.read_csv(state_csv_path)
            s_acc = df_s['strict_match'].mean() * 100
            a_acc = df_s['adjacent_match'].mean() * 100

            summary_records.append({
                "step_size": step_size,
                "grad_steps": grad_steps,
                "strict_accuracy": s_acc,
                "adjacent_accuracy": a_acc,
                "folder_path": state_folder_name
            })

df_summary = pd.DataFrame(summary_records)
# Rank configuration frameworks based on Strict Accuracy metrics
df_summary = df_summary.sort_values(by="strict_accuracy", ascending=False).reset_index(drop=True)

readme_content = "="*70 + "\n"
readme_content += " 🔍 THESIS GRID SEARCH EXPERIMENTAL METRIC LOG DIRECTORY\n"
readme_content += "="*70 + "\n"
readme_content += f"Execution Context Date : 2026\n"
readme_content += f"Calibration Split Size : 120 Prompts (Strictly Locked Balance Profile)\n"
readme_content += f"Target Judge Classifier: {JUDGE_MODEL_ID}\n"
readme_content += "Description:\n"
readme_content += "This directory contains the completely isolated baseline logging results across\n"
readme_content += "all 50 permutations of step sizes and gradient steps for the Pure PPLM architecture.\n"
readme_content += "Each folder stores granular logs, standard text summary files, and metrics diagnostics.\n"
readme_content += "="*70 + "\n\n"

readme_content += "🏆 TOP 5 PARAMETER CANDIDATE CONFIGURATIONS (Ranked by Strict Accuracy):\n"
readme_content += "-"*70 + "\n"
for idx in range(min(5, len(df_summary))):
    row = df_summary.iloc[idx]
    readme_content += f" Rank {idx+1}: Folder -> [{row['folder_path']}]\n"
    readme_content += f"        • Step Size        : {row['step_size']}\n"
    readme_content += f"        • Gradient Steps   : {row['grad_steps']}\n"
    readme_content += f"        • Strict Accuracy  : {row['strict_accuracy']:.2f}%\n"
    readme_content += f"        • Adjacent Accuracy: {row['adjacent_accuracy']:.2f}%\n"
    readme_content += "-"*70 + "\n"

readme_save_path = os.path.join(GRID_SEARCH_DIR, "README.txt")
with open(readme_save_path, "w") as f:
    f.write(readme_content)

print(f"\n✨ SUCCESS! The central evaluation summary file has been generated and saved to:\n -> {readme_save_path}")
print("\n" + readme_content)


✨ SUCCESS! The central evaluation summary file has been generated and saved to:
 -> /content/drive/MyDrive/Mohammd_Thesis/Results/PPLM_Pure/GRID_SEARCH/README.txt

 🔍 THESIS GRID SEARCH EXPERIMENTAL METRIC LOG DIRECTORY
Execution Context Date : 2026
Calibration Split Size : 120 Prompts (Strictly Locked Balance Profile)
Target Judge Classifier: MohammadKhosravi/roberta-large-cefr-classifier-JointLoss
Description:
This directory contains the completely isolated baseline logging results across
all 50 permutations of step sizes and gradient steps for the Pure PPLM architecture.
Each folder stores granular logs, standard text summary files, and metrics diagnostics.

🏆 TOP 5 PARAMETER CANDIDATE CONFIGURATIONS (Ranked by Strict Accuracy):
----------------------------------------------------------------------
 Rank 1: Folder -> [step_size_0.5_grad_step_20]
        • Step Size        : 0.5
        • Gradient Steps   : 20
        • Strict Accuracy  : 29.17%
        • Adjacent Accuracy: 58.33%
-

## Extended Grid Search — 70 Configurations

The initial search space is extended by adding four intermediate perturbation step sizes (`0.2`, `0.4`, `0.6`, and `0.8`), increasing the complete search space from 50 to 70 hyperparameter configurations.

In [ ]:
!pip install -q transformers torch datasets tqdm huggingface_hub accelerate scikit-learn textstat spacy
!python -m spacy download en_core_web_sm

import os
import gc
import torch
import spacy
import textstat
import pandas as pd
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
from huggingface_hub import login, hf_hub_download
from google.colab import drive

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 105.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 125.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# ==========================================
# 0. CONFIGURATION & GOOGLE DRIVE SETUP
# ==========================================
drive.mount('/content/drive')

INPUT_CSV_PATH = "/content/drive/MyDrive/Your_Path/in_domain_evaluation_prompt_matrix.csv"
BASE_OUTPUT_DIR = "/content/drive/MyDrive/Your_Path/pplm_grid_search_results"
GRID_SEARCH_DIR = os.path.join(BASE_OUTPUT_DIR, "GRID_SEARCH")
os.makedirs(GRID_SEARCH_DIR, exist_ok=True)

CALIBRATION_CSV_PATH = os.path.join(GRID_SEARCH_DIR, "grid_search_calibration_subset.csv")

hf_token = "hf_token"
login(token=hf_token)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load NLP structural tools
nlp = spacy.load("en_core_web_sm")

Mounted at /content/drive
Using device: cuda


In [ ]:
# ==========================================
# 1. GENERATE / LOAD PERMANENT CALIBRATION SPLIT
# ==========================================
df_master = pd.read_csv(INPUT_CSV_PATH)
df_master['cefr'] = df_master['cefr'].astype(str).str.strip().str.upper()

if os.path.exists(CALIBRATION_CSV_PATH):
    print(f"Loading existing permanent 120-prompt calibration subset from: {CALIBRATION_CSV_PATH}")
    df_calibration = pd.read_csv(CALIBRATION_CSV_PATH)
else:
    print("Extracting deterministic 120-prompt calibration subset (20 topics × 6 levels)...")
    # Group by title to identify topics that have variants for all 6 target categories
    grouped = df_master.groupby('topic_title').filter(lambda x: set(["A1", "A2", "B1", "B2", "C1", "C2"]).issubset(set(x['cefr'])))
    unique_topics = grouped['topic_title'].unique()

    if len(unique_topics) < 20:
        raise ValueError(f"Error: Found only {len(unique_topics)} unique topics containing all 6 levels. Expected at least 20.")

    # Pick exactly 20 unique topics deterministically using a fixed random state
    np.random.seed(42)
    selected_topics = np.random.choice(unique_topics, size=20, replace=False)

    # Filter master data to pull matching rows
    df_filtered = df_master[df_master['topic_title'].isin(selected_topics)]

    # Isolate exactly 1 row per CEFR level for each of the selected 20 topics to ensure exactly 120 rows
    final_rows = []
    for topic in selected_topics:
        topic_df = df_filtered[df_filtered['topic_title'] == topic]
        for level in ["A1", "A2", "B1", "B2", "C1", "C2"]:
            matched_row = topic_df[topic_df['cefr'] == level].head(1)
            final_rows.append(matched_row)

    df_calibration = pd.concat(final_rows, ignore_index=True)
    df_calibration.to_csv(CALIBRATION_CSV_PATH, index=False)
    print(f"Saved locked calibration split validation file to Drive: {CALIBRATION_CSV_PATH}")

print(f"Calibration Dataset Check -> Length: {len(df_calibration)} rows (Expected: 120)")

Loading existing permanent 120-prompt calibration subset from: /content/drive/MyDrive/Mohammd_Thesis/Results/PPLM_Pure/GRID_SEARCH/grid_search_calibration_subset.csv
Calibration Dataset Check -> Length: 120 rows (Expected: 120)


In [ ]:
# ==========================================
# 2. LOAD LLM & TOKENIZER
# ==========================================
print("Loading Llama-3.1-8B-Instruct...")
model_id = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>\n") if tokenizer.convert_tokens_to_ids("<|eot_id|>\n") is not None else tokenizer.convert_tokens_to_ids("<|eot_id|>. ")
if eot_id is None:
    eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
eos_ids = [tokenizer.eos_token_id]
if eot_id is not None:
    eos_ids.append(eot_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, device_map="auto"
)
model.eval()

Loading Llama-3.1-8B-Instruct...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
  

In [ ]:
# ==========================================
# 3. LOAD PPLM STEERING HEAD
# ==========================================
# =========================================================
# LOAD THE PPLM CEFR LATENT CLASSIFIER
# =========================================================

print("Loading PPLM CEFR latent classifier...")

class CEFRLatentClassifier(nn.Module):
    def __init__(self, input_dim=4096, num_classes=6):
        super().__init__()
        self.network = nn.Sequential(
            nn.Dropout(p=0.35),
            nn.Linear(input_dim, num_classes)
        )

    def forward(self, x):
        return self.network(x)

target_repo_id = (
    "MohammadKhosravi/"
    "llama3.1-8b-cefr-steering-1layer-head-ordinal-universal"
)

steering_head = CEFRLatentClassifier().to(device)

weights_path = hf_hub_download(
    repo_id=target_repo_id,
    filename="cefr_steering_head_universal_ordinal.pt"
)

steering_head.load_state_dict(
    torch.load(weights_path, map_location=device)
)

steering_head.eval()

criterion = nn.CrossEntropyLoss(reduction="none")

In [ ]:
# ==========================================
# 4. LOAD CUSTOM JOINT LOSS EVALUATOR
# ==========================================
print("Loading Custom JointLoss RoBERTa Evaluator...")
JUDGE_MODEL_ID = "MohammadKhosravi/roberta-large-cefr-classifier-JointLoss"
judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID)
judge_model = AutoModelForSequenceClassification.from_pretrained(JUDGE_MODEL_ID).to(device)
judge_model.eval()

label_map = {"A1": 0, "A2": 1, "B1": 2, "B2": 3, "C1": 4, "C2": 5}
inv_label_map = {0: "A1", 1: "A2", 2: "B1", 3: "B2", 4: "C1", 5: "C2"}

Loading Custom JointLoss RoBERTa Evaluator...


config.json:   0%|          | 0.00/900 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/388 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [ ]:
# ==========================================
# 5. METRIC HELPER FUNCTIONS
# ==========================================
def calculate_mdd(text):
    doc = nlp(text)
    total_dist, tokens = 0, 0
    for token in doc:
        if token.dep_ != "punct":
            total_dist += abs(token.i - token.head.i)
            tokens += 1
    return total_dist / tokens if tokens > 0 else 0.0

def calculate_sentence_drift(text, judge_model, judge_tokenizer):
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents if len(sent.text.strip()) > 10]
    if len(sentences) <= 1: return 0

    inputs = judge_tokenizer(sentences, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        preds = torch.argmax(judge_model(**inputs).logits, dim=-1).cpu().numpy()

    return int(np.max(preds) - np.min(preds))

In [ ]:
# ==========================================
# 6. GRID SEARCH EXECUTION PARAMETERS (70 STATES TOTAL)
# ==========================================
# Includes original 10 + new 4 (0.2, 0.4, 0.6, 0.8) -> 14 total step sizes
step_sizes = [0.01, 0.02, 0.05, 0.07, 0.09, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
grad_steps_list = [5, 10, 15, 20, 25]

BATCH_SIZE = 1024
temperature = 0.6
repetition_penalty = 1.15
max_new_tokens = 200
top_k = 50

print(f"\n🚀 STARTING GRID SEARCH ROUTINE ({len(step_sizes) * len(grad_steps_list)} total permutations)...")
print("ℹ️ Previously computed states will be automatically skipped.")

for step_size in step_sizes:
    for grad_steps in grad_steps_list:

        # Define target folder for this hyperparameter permutation state
        state_folder_name = f"step_size_{step_size}_grad_step_{grad_steps}"
        state_dir = os.path.join(GRID_SEARCH_DIR, state_folder_name)
        os.makedirs(state_dir, exist_ok=True)

        state_csv_path = os.path.join(state_dir, "benchmark_results.csv")
        state_txt_path = os.path.join(state_dir, "metrics_report.txt")
        state_img_path = os.path.join(state_dir, "confusion_matrix.png")

        # Skip logic: If CSV and TXT exist, it's already done!
        if os.path.exists(state_txt_path) and os.path.exists(state_csv_path):
            print(f"⏭️ Skipping configuration state [Step: {step_size} | Grads: {grad_steps}] - Already complete.")
            continue

        print(f"\n⚙️ Executing Configuration Matrix State: Step Size = {step_size}, Grad Steps = {grad_steps}")
        results = []

        # Batch generation across our locked 120 calibration rows
        for b_start in range(0, len(df_calibration), BATCH_SIZE):
            batch_df = df_calibration.iloc[b_start : b_start + BATCH_SIZE]
            current_batch_size = len(batch_df)

            batch_prompts, batch_targets, batch_target_ids, batch_topic_ids, batch_titles = [], [], [], [], []

            for _, row in batch_df.iterrows():
                t_cefr = str(row['cefr']).strip().upper()
                batch_targets.append(t_cefr)
                batch_target_ids.append(label_map[t_cefr])
                batch_topic_ids.append(row['topic_id'])
                batch_titles.append(row['topic_title'])

                sys_instr = (
                    f"You are an expert English language teacher demonstrating CEFR proficiency levels. "
                    f"Your task is to write a flawless, grammatically correct text responding to this prompt: '{row['topic_title']}'. "
                    f"The output must serve as a perfect textbook example of strictly {t_cefr} level English. "
                    f"If the requested target level is A1/A2, use very simple vocabulary, short sentences, and primitive structures. "
                    f"If the requested target level is C1/C2, utilize highly advanced vocabulary, idioms, and complex sentence patterns. "
                    f"Write only the direct response. Do not write any meta-commentary, greetings, or conversational pleasantries."
                )
                messages = [{"role": "system", "content": sys_instr}, {"role": "user", "content": row['topic_title']}]
                batch_prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

            inputs = tokenizer(batch_prompts, padding=True, return_tensors="pt").to(device)
            input_ids, attention_mask = inputs.input_ids, inputs.attention_mask

            finished = torch.zeros(current_batch_size, dtype=torch.bool, device=device)
            generated_tokens = [[] for _ in range(current_batch_size)]
            batch_target_ids_tensor = torch.tensor(batch_target_ids, device=device)

            past_key_values = None
            current_attention_mask = attention_mask

            # Autoregressive PPLM Fixed Steering Loop Execution
            for step_idx in range(max_new_tokens):
                model_input = input_ids if step_idx == 0 else next_token_ids.unsqueeze(-1)
                if step_idx > 0:
                    current_attention_mask = torch.cat([current_attention_mask, torch.ones(current_batch_size, 1, dtype=torch.long, device=device)], dim=-1)

                with torch.no_grad():
                    outputs = model(
                        input_ids=model_input, attention_mask=current_attention_mask,
                        past_key_values=past_key_values, use_cache=True, output_hidden_states=True
                    )
                past_key_values = outputs.past_key_values

                hidden_state_prenorm = outputs.hidden_states[-2][:, -1, :].detach().clone().to(torch.float32)
                hidden_state_prenorm.requires_grad_(True)

                for g_step in range(grad_steps):
                    class_logits = steering_head(hidden_state_prenorm)
                    loss_vector = criterion(class_logits, batch_target_ids_tensor)

                    active_mask = (~finished).float()
                    loss = (loss_vector * active_mask).sum()
                    if loss.item() == 0: break

                    loss.backward()

                    with torch.no_grad():
                        raw_grad = hidden_state_prenorm.grad.data
                        grad_norms = torch.norm(raw_grad, p=2, dim=-1, keepdim=True) + 1e-9
                        normalized_grad = raw_grad / grad_norms

                        hidden_state_prenorm.data = hidden_state_prenorm.data - (step_size * normalized_grad * active_mask.unsqueeze(-1))
                        hidden_state_prenorm.grad.zero_()

                # Softmax mapping to vocab logit spaces
                with torch.no_grad():
                    steered_normed = model.model.norm(hidden_state_prenorm.to(torch.bfloat16))
                    lm_logits = model.lm_head(steered_normed)

                    for b_idx in range(current_batch_size):
                        for tok_id in set(generated_tokens[b_idx] + input_ids[b_idx].tolist()):
                            if lm_logits[b_idx, tok_id] > 0: lm_logits[b_idx, tok_id] /= repetition_penalty
                            else: lm_logits[b_idx, tok_id] *= repetition_penalty

                    lm_logits = lm_logits / temperature
                    values, indices = torch.topk(lm_logits, top_k, dim=-1)
                    filtered_logits = torch.full_like(lm_logits, float('-inf')).scatter_(1, indices, values)
                    probs_lm = F.softmax(filtered_logits, dim=-1)

                    next_token_ids = torch.multinomial(probs_lm, num_samples=1).squeeze(-1)

                    for b_idx in range(current_batch_size):
                        if not finished[b_idx]:
                            generated_tokens[b_idx].append(next_token_ids[b_idx].item())
                            if next_token_ids[b_idx].item() in eos_ids:
                                finished[b_idx] = True

                if finished.all(): break

            batch_gen_texts = [tokenizer.decode(generated_tokens[b_idx], skip_special_tokens=True).strip() for b_idx in range(current_batch_size)]

            # Evaluation and structural post-processing across batch partitions
            eval_inputs = judge_tokenizer(batch_gen_texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
            with torch.no_grad():
                eval_preds = torch.argmax(judge_model(**eval_inputs).logits, dim=-1).cpu().numpy()

            for b_idx in range(current_batch_size):
                target_cefr = batch_targets[b_idx]
                predicted_cefr = inv_label_map.get(eval_preds[b_idx], "A1")
                gen_text = batch_gen_texts[b_idx]

                is_strict = int(predicted_cefr == target_cefr)
                is_adjacent = int(abs(label_map[predicted_cefr] - label_map[target_cefr]) <= 1)

                try: flesch_score = textstat.flesch_reading_ease(gen_text)
                except: flesch_score = 0.0

                mdd_score = calculate_mdd(gen_text)
                drift_score = calculate_sentence_drift(gen_text, judge_model, judge_tokenizer)

                results.append({
                    "topic_id": batch_topic_ids[b_idx], "topic_title": batch_titles[b_idx],
                    "target_cefr": target_cefr, "predicted_cefr": predicted_cefr,
                    "strict_match": is_strict, "adjacent_match": is_adjacent,
                    "mdd": round(mdd_score, 2), "readability_flesch": round(flesch_score, 2),
                    "sentence_drift_max": drift_score, "generated_text": gen_text
                })

            del inputs, eval_inputs
            torch.cuda.empty_cache()
            gc.collect()

        # Save current state data tracking matrix frames
        df_state_results = pd.DataFrame(results)
        df_state_results.to_csv(state_csv_path, index=False)

        # Macro analysis calculations
        y_true_labels = df_state_results['target_cefr'].astype(str).str.strip().str.upper()
        y_pred_labels = df_state_results['predicted_cefr'].astype(str).str.strip().str.upper()
        y_true_num = y_true_labels.map(label_map)
        y_pred_num = y_pred_labels.map(label_map)

        strict_acc = df_state_results['strict_match'].mean() * 100
        adj_acc = df_state_results['adjacent_match'].mean() * 100
        mae = mean_absolute_error(y_true_num, y_pred_num)
        avg_mdd = df_state_results['mdd'].mean()
        avg_flesch = df_state_results['readability_flesch'].mean()
        avg_drift = df_state_results['sentence_drift_max'].mean()

        report_text = "="*60 + "\n"
        report_text += f" 📊 METRICS REPORT [Step: {step_size} | Grads: {grad_steps}]\n"
        report_text += "="*60 + "\n"
        report_text += f"Total Processed       : {len(df_state_results)}\n"
        report_text += f"Strict Accuracy       : {strict_acc:.2f}%\n"
        report_text += f"Adjacent Accuracy     : {adj_acc:.2f}%\n"
        report_text += f"Mean Abs Error (MAE)  : {mae:.4f}\n"
        report_text += f"Avg MDD Score         : {avg_mdd:.2f}\n"
        report_text += f"Avg Reading Ease      : {avg_flesch:.2f}\n"
        report_text += f"Avg Sentence Drift    : {avg_drift:.2f} levels\n"
        report_text += "="*60 + "\n\n"

        report_text += "📈 PERFORMANCE BREAKDOWN BY CEFR TARGET LEVEL:\n"
        level_agg = df_state_results.groupby("target_cefr").agg(
            strict_accuracy=("strict_match", lambda x: np.mean(x) * 100),
            avg_mdd=("mdd", "mean"),
            avg_flesch=("readability_flesch", "mean"),
            avg_drift=("sentence_drift_max", "mean")
        ).round(2)
        report_text += level_agg.to_string() + "\n\n"

        report_text += "="*60 + "\n"
        report_text += "📝 DETAILED CLASSIFICATION REPORT:\n"
        cefr_labels = ["A1", "A2", "B1", "B2", "C1", "C2"]
        report_text += classification_report(y_true_labels, y_pred_labels, labels=cefr_labels, zero_division=0)
        report_text += "\n" + "="*60 + "\n"

        with open(state_txt_path, "w") as f:
            f.write(report_text)

        # Plot confusion validation matrices
        cm = confusion_matrix(y_true_labels, y_pred_labels, labels=cefr_labels)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=cefr_labels, yticklabels=cefr_labels, cbar=True, square=True)
        plt.title(f"CEFR Confusion Matrix (Step {step_size} | Grads {grad_steps})", fontsize=11, pad=12)
        plt.xlabel("Predicted CEFR Level", fontsize=9)
        plt.ylabel("Target CEFR Level", fontsize=9)
        plt.tight_layout()
        plt.savefig(state_img_path, dpi=300)
        plt.close()

print("\n🏁 ALL 70 STATES PROCESSED. COMPILING MASTER SUMMARY PROFILE...")



🚀 STARTING GRID SEARCH ROUTINE (70 total permutations)...
ℹ️ Previously computed states will be automatically skipped.
⏭️ Skipping configuration state [Step: 0.01 | Grads: 5] - Already complete.
⏭️ Skipping configuration state [Step: 0.01 | Grads: 10] - Already complete.
⏭️ Skipping configuration state [Step: 0.01 | Grads: 15] - Already complete.
⏭️ Skipping configuration state [Step: 0.01 | Grads: 20] - Already complete.
⏭️ Skipping configuration state [Step: 0.01 | Grads: 25] - Already complete.
⏭️ Skipping configuration state [Step: 0.02 | Grads: 5] - Already complete.
⏭️ Skipping configuration state [Step: 0.02 | Grads: 10] - Already complete.
⏭️ Skipping configuration state [Step: 0.02 | Grads: 15] - Already complete.
⏭️ Skipping configuration state [Step: 0.02 | Grads: 20] - Already complete.
⏭️ Skipping configuration state [Step: 0.02 | Grads: 25] - Already complete.
⏭️ Skipping configuration state [Step: 0.05 | Grads: 5] - Already complete.
⏭️ Skipping configuration state [Ste

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



⚙️ Executing Configuration Matrix State: Step Size = 0.2, Grad Steps = 10

⚙️ Executing Configuration Matrix State: Step Size = 0.2, Grad Steps = 15

⚙️ Executing Configuration Matrix State: Step Size = 0.2, Grad Steps = 20

⚙️ Executing Configuration Matrix State: Step Size = 0.2, Grad Steps = 25
⏭️ Skipping configuration state [Step: 0.3 | Grads: 5] - Already complete.
⏭️ Skipping configuration state [Step: 0.3 | Grads: 10] - Already complete.
⏭️ Skipping configuration state [Step: 0.3 | Grads: 15] - Already complete.
⏭️ Skipping configuration state [Step: 0.3 | Grads: 20] - Already complete.
⏭️ Skipping configuration state [Step: 0.3 | Grads: 25] - Already complete.

⚙️ Executing Configuration Matrix State: Step Size = 0.4, Grad Steps = 5

⚙️ Executing Configuration Matrix State: Step Size = 0.4, Grad Steps = 10

⚙️ Executing Configuration Matrix State: Step Size = 0.4, Grad Steps = 15

⚙️ Executing Configuration Matrix State: Step Size = 0.4, Grad Steps = 20

⚙️ Executing Configur

In [ ]:
# ==========================================
# 7. AUTOMATED CENTRAL SUMMARY FILE GENERATION
# ==========================================
summary_records = []
for step_size in step_sizes:
    for grad_steps in grad_steps_list:
        state_folder_name = f"step_size_{step_size}_grad_step_{grad_steps}"
        state_csv_path = os.path.join(GRID_SEARCH_DIR, state_folder_name, "benchmark_results.csv")

        if os.path.exists(state_csv_path):
            df_s = pd.read_csv(state_csv_path)
            s_acc = df_s['strict_match'].mean() * 100
            a_acc = df_s['adjacent_match'].mean() * 100

            summary_records.append({
                "step_size": step_size,
                "grad_steps": grad_steps,
                "strict_accuracy": s_acc,
                "adjacent_accuracy": a_acc,
                "folder_path": state_folder_name
            })

df_summary = pd.DataFrame(summary_records)
# Rank configuration frameworks based on Strict Accuracy metrics
df_summary = df_summary.sort_values(by="strict_accuracy", ascending=False).reset_index(drop=True)

readme_content = "="*70 + "\n"
readme_content += " 🔍 THESIS GRID SEARCH EXPERIMENTAL METRIC LOG DIRECTORY (70 STATES)\n"
readme_content += "="*70 + "\n"
readme_content += f"Execution Context Date : 2026\n"
readme_content += f"Calibration Split Size : 120 Prompts (Strictly Locked Balance Profile)\n"
readme_content += f"Target Judge Classifier: {JUDGE_MODEL_ID}\n"
readme_content += "Description:\n"
readme_content += "This directory contains the completely isolated baseline logging results across\n"
readme_content += "all 70 permutations of step sizes and gradient steps for the Pure PPLM architecture.\n"
readme_content += "Each folder stores granular logs, standard text summary files, and metrics diagnostics.\n"
readme_content += "="*70 + "\n\n"

readme_content += "🏆 TOP 5 PARAMETER CANDIDATE CONFIGURATIONS (Ranked by Strict Accuracy):\n"
readme_content += "-"*70 + "\n"
for idx in range(min(5, len(df_summary))):
    row = df_summary.iloc[idx]
    readme_content += f" Rank {idx+1}: Folder -> [{row['folder_path']}]\n"
    readme_content += f"        • Step Size        : {row['step_size']}\n"
    readme_content += f"        • Gradient Steps   : {row['grad_steps']}\n"
    readme_content += f"        • Strict Accuracy  : {row['strict_accuracy']:.2f}%\n"
    readme_content += f"        • Adjacent Accuracy: {row['adjacent_accuracy']:.2f}%\n"
    readme_content += "-"*70 + "\n"

readme_save_path = os.path.join(GRID_SEARCH_DIR, "README.txt")
with open(readme_save_path, "w") as f:
    f.write(readme_content)

print(f"\n✨ SUCCESS! The central evaluation summary file has been generated and saved to:\n -> {readme_save_path}")
print("\n" + readme_content)

# ---------------------------------------------------------
# 🚨 CHECK FOR NEW BEST CANDIDATE (SURPASSING PREVIOUS 29.17%)
# ---------------------------------------------------------
highest_strict_acc = df_summary.iloc[0]['strict_accuracy']
best_folder = df_summary.iloc[0]['folder_path']

if highest_strict_acc > 29.17:
    print("\n" + "🔥" * 25)
    print("🚨 ATTENTION: NEW HIGHEST PERFORMING CANDIDATE FOUND! 🚨")
    print(f"The state '{best_folder}' achieved {highest_strict_acc:.2f}% Strict Accuracy.")
    print("This surpasses the previous baseline maximum of 29.17%!")
    print("You may need to run full-scale evaluations on this new winner.")
    print("🔥" * 25 + "\n")
else:
    print("\n✔️ UPDATE COMPLETE: No new combinations surpassed the previous 29.17% strict accuracy ceiling.")
    print("The grid search conclusion remains unchanged.")


✨ SUCCESS! The central evaluation summary file has been generated and saved to:
 -> /content/drive/MyDrive/Mohammd_Thesis/Results/PPLM_Pure/GRID_SEARCH/README.txt

 🔍 THESIS GRID SEARCH EXPERIMENTAL METRIC LOG DIRECTORY (70 STATES)
Execution Context Date : 2026
Calibration Split Size : 120 Prompts (Strictly Locked Balance Profile)
Target Judge Classifier: MohammadKhosravi/roberta-large-cefr-classifier-JointLoss
Description:
This directory contains the completely isolated baseline logging results across
all 70 permutations of step sizes and gradient steps for the Pure PPLM architecture.
Each folder stores granular logs, standard text summary files, and metrics diagnostics.

🏆 TOP 5 PARAMETER CANDIDATE CONFIGURATIONS (Ranked by Strict Accuracy):
----------------------------------------------------------------------
 Rank 1: Folder -> [step_size_0.5_grad_step_20]
        • Step Size        : 0.5
        • Gradient Steps   : 20
        • Strict Accuracy  : 29.17%
        • Adjacent Accura